In [ ]:
import json

# Open and read the JSON file
with open('dqd_results.json', 'r') as file:
    data = json.load(file)

# Check if 'CheckResults' is in the data
if 'CheckResults' in data:
    for result in data['CheckResults']:
        # Remove the 'QUERY_TEXT' key if it exists in each item
        if 'QUERY_TEXT' in result:
            del result['QUERY_TEXT']

# Write the modified JSON back to a new file
with open('modified_dqd_results.json', 'w') as file:
    json.dump(data, file, indent=4)

print("The 'QUERY_TEXT' field has been removed successfully.")

In [ ]:
import pandas as pd
from google.cloud import bigquery

In [ ]:
with open('modified_dqd_results.json', 'r') as file:
    data = json.load(file)
    if 'CheckResults' in data:
        extracted_data = [
            {
                'HPO_ID': result.get('HPO_ID'),
                'CHECK_NAME': result.get('CHECK_NAME'),
                'CHECK_LEVEL': result.get('CHECK_LEVEL'),
                'CDM_TABLE_NAME': result.get('CDM_TABLE_NAME'),
                'CATEGORY': result.get('CATEGORY'),
                'SUBCATEGORY': result.get('SUBCATEGORY'),
                'FAILED': result.get('FAILED'),
                'PASSED': result.get('PASSED'),
                'IS_ERROR': result.get('IS_ERROR'),
                'NOT_APPLICABLE': result.get('NOT_APPLICABLE')
            }
            for result in data['CheckResults']
        ]
    
        # Convert the list of dictionaries to a DataFrame
        df = pd.DataFrame(extracted_data)
    
        # Display the DataFrame
        print(df)
    else:
        print("CheckResults not found in the JSON data.")

In [ ]:
df_results = df.groupby(by=['HPO_ID', 'CATEGORY']).sum()[['PASSED', 'FAILED', 'NOT_APPLICABLE']].reset_index()

In [ ]:
df_results['PASSED_RATE'] = df_results['PASSED'] / (df_results['PASSED'] + df_results['FAILED'] + df_results['NOT_APPLICABLE'])
df_results['FAILED_RATE'] = df_results['FAILED'] / (df_results['PASSED'] + df_results['FAILED'] + df_results['NOT_APPLICABLE'])
df_results['NOT_APPLICABLE_RATE'] = df_results['NOT_APPLICABLE'] / (df_results['PASSED'] + df_results['FAILED'] + df_results['NOT_APPLICABLE'])

In [ ]:
df_results

In [ ]:
from utils import read_config

config = read_config.load_config()

In [ ]:
namespaces = config['bigquery']['namespaces']
OUTPUT_NAMESPACE = namespaces['output']
MAPPING_NAMESPACE = namespaces['mappings']
OMOP_TABLE_PREFIX = config['omop_table_configs']['OMOP_TABLE_PREFIX']
COHORT_TABLE = config['analysis_configs']['COHORT_TABLE']
COHORT_LABEL_PREFIX = config['analysis_configs']['COHORT_LABEL_PREFIX']
ANALYSIS_END_DATE = config['omop_table_configs']['ANALYSIS_END_DATE']

In [ ]:
loyalty_score_query = f"""
SELECT sm.hpo_id AS src_hpo_id, ls.patient_num_pid, ls.race, ls.ethnicity, ls.gender, ls.age, ls.index_dt, ls.last_visit, ls.score, ls.cohort_name
FROM `{OUTPUT_NAMESPACE}.{COHORT_TABLE}` AS ls
LEFT JOIN `{MAPPING_NAMESPACE}.site_maskings` AS sm
ON ls.src_id = sm.src_id
"""

In [ ]:
client = bigquery.Client()

In [ ]:
ls_df = client.query(loyalty_score_query).to_dataframe()

In [ ]:
ls_by_hpo = ls_df[ls_df['cohort_name'] == f'{COHORT_LABEL_PREFIX}_{ANALYSIS_END_DATE}_1years'].groupby(['src_hpo_id'])['score'].median().reset_index()

In [ ]:
ls_by_hpo.rename({'score': 'median_loyalty_score'}, axis=1, inplace=True)

In [ ]:
ls_by_hpo

In [ ]:
loyalty_dqd_score = pd.merge(ls_by_hpo, df_results, how='left', left_on=['src_hpo_id'], right_on=['HPO_ID'])

In [ ]:
loyalty_dqd_score = loyalty_dqd_score[['src_hpo_id', 'median_loyalty_score', 'CATEGORY', 'PASSED_RATE']]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
ax = sns.scatterplot(
    data=loyalty_dqd_score,
    x='median_loyalty_score',
    y='PASSED_RATE',
    hue='CATEGORY'
)

ax.set(
    xlabel='Median Loyalty Score with 1 Lookback Year',
    ylabel='Data Quality Score',
    title='Loyalty Score and Data Quality Score by HPO Site'
)

plt.legend(title='Data Quality Check Category', loc='upper left')

plt.savefig("loyalty_score_dqd_1year.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
loyalty_dqd_score.groupby('CATEGORY').size()

## Pearson correlations by DQD dimension

Pearson correlations are calculated across HPO sites between the median 5-year loyalty score and each DQD category's pass rate. The 95% confidence intervals use Fisher's *z* transformation. Rows with a missing loyalty score, DQD category, or pass rate are excluded separately for each dimension.

In [ ]:
import numpy as np

def pearson_with_fisher_ci(data, category, confidence_level=0.95):
    category_data = (
        data.loc[
            data['CATEGORY'].eq(category),
            ['src_hpo_id', 'median_loyalty_score', 'PASSED_RATE']
        ]
        .dropna()
        .drop_duplicates(subset='src_hpo_id')
    )
    n = len(category_data)
    if n < 4:
        raise ValueError(f'{category}: at least 4 complete HPO observations are required for a Fisher-z CI; found {n}.')
    if category_data['median_loyalty_score'].nunique() < 2 or category_data['PASSED_RATE'].nunique() < 2:
        raise ValueError(f'{category}: Pearson correlation is undefined because one variable is constant.')

    r = category_data['median_loyalty_score'].corr(category_data['PASSED_RATE'], method='pearson')
    if confidence_level != 0.95:
        raise ValueError('This implementation currently supports a 95% confidence interval only.')
    z_critical = 1.959963984540054
    z = np.arctanh(np.clip(r, -1 + np.finfo(float).eps, 1 - np.finfo(float).eps))
    margin = z_critical / np.sqrt(n - 3)
    ci_low, ci_high = np.tanh([z - margin, z + margin])

    return {
        'DQD dimension': category,
        'HPO sites (n)': n,
        'Pearson r': r,
        '95% CI lower': ci_low,
        '95% CI upper': ci_high,
    }

dqd_dimensions = ['Completeness', 'Conformance', 'Plausibility']
pearson_results = pd.DataFrame([
    pearson_with_fisher_ci(loyalty_dqd_score, dimension)
    for dimension in dqd_dimensions
])
pearson_results

In [ ]:
pearson_reporting_table = pearson_results.copy()
pearson_reporting_table['Pearson r (95% CI)'] = pearson_reporting_table.apply(
    lambda row: f"{row['Pearson r']:.3f} ({row['95% CI lower']:.3f} to {row['95% CI upper']:.3f})",
    axis=1,
)
pearson_reporting_table = pearson_reporting_table[
    ['DQD dimension', 'HPO sites (n)', 'Pearson r (95% CI)']
]
pearson_reporting_table

In [ ]:
correlation_statements = [
    (
        f"{row['DQD dimension']} (r={row['Pearson r']:.3f}, "
        f"95% CI {row['95% CI lower']:.3f} to {row['95% CI upper']:.3f}; "
        f"n={int(row['HPO sites (n)'])} HPOs)"
    )
    for _, row in pearson_results.iterrows()
]
print(
    'Across HPO sites, Pearson correlations between the median 5-year loyalty score '
    'and DQD pass rates were: ' + '; '.join(correlation_statements) + '.'
)